# Technology Recommendation Engine

This notebook builds a data-informed technology recommender using technology co-occurrence patterns and similar employed developer profiles.

In [123]:
import pandas as pd

df_raw = pd.read_csv(
    "../../data/raw/stackoverflow_full.csv"
)

In [124]:
df_clean = (
    df_raw
    .drop(columns=["Unnamed: 0"], errors="ignore")
    .drop_duplicates()
    .copy()
)

recommender_mask = (
    (df_clean["Employed"] == 1)
    & (df_clean["YearsCodePro"] <= df_clean["YearsCode"])
    & (df_clean["PreviousSalary"] >= 1000)
    & (df_clean["HaveWorkedWith"].notna())
)

recommender_columns = [
    "Country",
    "EdLevel",
    "YearsCodePro",
    "PreviousSalary",
    "HaveWorkedWith",
    "ComputerSkills",
    "YearsCode"
]

recommender_df = df_clean.loc[
    recommender_mask,
    recommender_columns
].copy()

recommender_df.shape

(38823, 7)

In [125]:
recommender_df["TechnologyList"] = recommender_df["HaveWorkedWith"].str.split(";")

recommender_df[["TechnologyList", "HaveWorkedWith"]].head()

,TechnologyList,HaveWorkedWith
1,"[Bash/Shell, HTML/CSS, JavaScript, Node.js, SQ...",Bash/Shell;HTML/CSS;JavaScript;Node.js;SQL;Typ...
6,"[C++, HTML/CSS, Java, JavaScript, Kotlin, Node...",C++;HTML/CSS;Java;JavaScript;Kotlin;Node.js;Ty...
10,"[Bash/Shell, Go, Java, Node.js, Python, Scala,...",Bash/Shell;Go;Java;Node.js;Python;Scala;SQL;Do...
11,"[Assembly, C, C#, C++, HTML/CSS, Java, JavaScr...",Assembly;C;C#;C++;HTML/CSS;Java;JavaScript;Mat...
15,"[C#, C++, JavaScript, PowerShell, SQL, TypeScr...",C#;C++;JavaScript;PowerShell;SQL;TypeScript;Do...


In [126]:
exploded_technologies = recommender_df["TechnologyList"].explode()
exploded_technologies

1                  Bash/Shell
1                    HTML/CSS
1                  JavaScript
1                     Node.js
1                         SQL
                 ...         
73460                React.js
73460                     AWS
73460                DynamoDB
73460    Microsoft SQL Server
73460                  SQLite
Name: TechnologyList, Length: 669742, dtype: object

In [127]:
technology_counts = exploded_technologies.value_counts()

In [128]:
technology_counts.head(20)

TechnologyList
JavaScript              33236
HTML/CSS                27509
SQL                     25099
Docker                  24134
TypeScript              22762
Node.js                 22271
MySQL                   18670
AWS                     18558
Git                     18283
PostgreSQL              17877
React.js                17513
C#                      17014
Python                  16489
Microsoft SQL Server    16099
Java                    15789
jQuery                  15063
Bash/Shell              15042
MongoDB                 14510
SQLite                  14390
npm                     14312
Name: count, dtype: int64

In [129]:
exploded_technologies.nunique()

116

In [130]:
from sklearn.preprocessing import MultiLabelBinarizer

technology_encoder = MultiLabelBinarizer()

In [131]:
encoded_technologies = technology_encoder.fit_transform(recommender_df["TechnologyList"])

encoded_technologies

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 1, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 1, 1, 1]])

In [132]:
technology_matrix = pd.DataFrame(
    encoded_technologies,
    columns=technology_encoder.classes_,
    index=recommender_df.index
)

technology_matrix.head()

,APL,ASP.NET,ASP.NET Core,AWS,Angular,Angular.js,Ansible,Assembly,Bash/Shell,Blazor,C,C#,C++,COBOL,Cassandra,Chef,Clojure,Cloud Firestore,Colocation,CouchDB,Couchbase,Crystal,Dart,Delphi,Deno,DigitalOcean,Django,Docker,Drupal,DynamoDB,Elasticsearch,Elixir,Erlang,Express,F#,FastAPI,Fastify,Firebase,Firebase Realtime Database,Flask,...,OVH,Objective-C,OpenStack,Oracle,Oracle Cloud Infrastructure,PHP,Perl,Phoenix,Play Framework,PostgreSQL,PowerShell,Pulumi,Puppet,Python,R,React.js,Redis,Ruby,Ruby on Rails,Rust,SAS,SQL,SQLite,Scala,Solidity,Spring,Svelte,Swift,Symfony,Terraform,TypeScript,Unity 3D,Unreal Engine,VBA,VMware,Vue.js,Xamarin,Yarn,jQuery,npm
1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0
6,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
10,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
11,0,0,0,0,0,0,0,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0
15,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0


In [133]:
technology_matrix.shape

(38823, 116)

In [134]:
cooccurrence_matrix = technology_matrix.T @ technology_matrix

In [135]:
cooccurrence_matrix.Python.sort_values(ascending=False).head(15)

Python        16489
JavaScript    14074
Docker        11787
HTML/CSS      11701
SQL           11290
Node.js        9974
PostgreSQL     9416
AWS            9108
TypeScript     8942
MySQL          8882
Bash/Shell     8690
Git            8110
Java           8005
SQLite         7852
React.js       7764
Name: Python, dtype: int64

In [136]:
from sklearn.metrics.pairwise import cosine_similarity

In [137]:
technology_similarity_array = cosine_similarity(technology_matrix.T.astype(float))

/Users/ccakir/Desktop/codepath_ai/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ccakir/Desktop/codepath_ai/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ccakir/Desktop/codepath_ai/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [138]:
technology_similarity_matrix = pd.DataFrame(technology_similarity_array, columns=technology_matrix.columns, index=technology_matrix.columns)

In [139]:
technology_similarity_matrix.Python.sort_values(ascending=False).head(15)

Python        1.000000
JavaScript    0.601196
Docker        0.590869
SQL           0.554969
Bash/Shell    0.551785
HTML/CSS      0.549400
PostgreSQL    0.548431
AWS           0.520667
Node.js       0.520478
SQLite        0.509745
MySQL         0.506223
Flask         0.505783
Java          0.496120
Django        0.472938
Git           0.467089
Name: Python, dtype: float64

In [140]:
import numpy as np

In [141]:
technology_frequencies = np.diag(
    cooccurrence_matrix
)

frequency_pairs = np.outer(
    technology_frequencies,
    technology_frequencies
)

similarity_denominator = np.sqrt(
    frequency_pairs
)

technology_similarity_array = (
    cooccurrence_matrix.to_numpy()
    / similarity_denominator
)

technology_similarity_matrix = pd.DataFrame(
    technology_similarity_array,
    columns=technology_matrix.columns,
    index=technology_matrix.columns
)

technology_similarity_matrix.Python.sort_values(ascending=False).head(15)

Python        1.000000
JavaScript    0.601196
Docker        0.590869
SQL           0.554969
Bash/Shell    0.551785
HTML/CSS      0.549400
PostgreSQL    0.548431
AWS           0.520667
Node.js       0.520478
SQLite        0.509745
MySQL         0.506223
Flask         0.505783
Java          0.496120
Django        0.472938
Git           0.467089
Name: Python, dtype: float64

In [142]:
sample_user_technologies = ["Python", "FastAPI", "React.js", "PostgreSQL"]

In [143]:
candidate_similarity_scores = (
    technology_similarity_matrix[
        sample_user_technologies
    ]
    .mean(axis=1)
)

In [144]:
recommendation_candidates = (
    candidate_similarity_scores
    .drop(index=sample_user_technologies)
    .sort_values(ascending=False)
)

recommendation_candidates.head(15)

JavaScript    0.527917
Docker        0.516275
Node.js       0.479101
HTML/CSS      0.474151
AWS           0.463673
SQL           0.455249
TypeScript    0.452760
MySQL         0.414426
Bash/Shell    0.412580
MongoDB       0.401918
SQLite        0.394781
Java          0.383717
Redis         0.381324
Yarn          0.378220
Git           0.377742
dtype: float64

In [145]:
from pathlib import Path
import joblib

In [146]:
project_root = Path.cwd().parents[1]

salary_artifact_directory = (
    project_root / "artifacts" / "salary" / "salary_model_bundle_v1.joblib"
)

salary_model_bundle = joblib.load(salary_artifact_directory)

In [147]:
salary_model_bundle["features"]

['Country', 'EdLevel', 'YearsCode', 'YearsCodePro', 'ComputerSkills']

In [148]:
salary_profile_df = recommender_df[salary_model_bundle["features"]]

In [149]:
recommender_df["ExpectedSalary"] = salary_model_bundle["point_model"].predict(salary_profile_df)

recommender_df["SalaryResidual"] = recommender_df["PreviousSalary"] - recommender_df["ExpectedSalary"]

In [150]:
technology_salary_rows = (
    recommender_df[["TechnologyList", "SalaryResidual"]]
    .explode("TechnologyList")
    .rename(columns={"TechnologyList": "Technology"})
)

technology_salary_profile = (
    technology_salary_rows
    .groupby("Technology")
    .agg(
        developer_count=("SalaryResidual", "count"),
        mean_salary_residual=("SalaryResidual", "mean"),
        median_salary_residual=("SalaryResidual", "median"),
    )
    .query("developer_count >= 200")
    .sort_values("median_salary_residual", ascending=False)
    .round(2)
)

technology_salary_profile.head(15)

,developer_count,mean_salary_residual,median_salary_residual
Technology,,,
Pulumi,315,22160.97,15498.82
Phoenix,276,19066.02,14957.07
Terraform,4483,16559.10,10319.22
Homebrew,5159,14624.01,8311.27
Clojure,558,14278.52,7760.22
Chef,610,13795.82,7463.91
Scala,1192,14633.98,7337.15
Fastify,452,15310.46,7275.50
Solidity,305,15026.37,6839.41


In [152]:
technology_salary_profile["salary_score"] = (
    technology_salary_profile["median_salary_residual"]
    .rank(pct=True)
)

In [153]:
technology_salary_profile.head()

,developer_count,mean_salary_residual,median_salary_residual,salary_score
Technology,,,,
Pulumi,315,22160.97,15498.82,1.000000
Phoenix,276,19066.02,14957.07,0.990909
Terraform,4483,16559.10,10319.22,0.981818
Homebrew,5159,14624.01,8311.27,0.972727
Clojure,558,14278.52,7760.22,0.963636


In [155]:
final_recommendation_scores = recommendation_candidates.rename("similarity_score").to_frame()

In [158]:
final_recommendation_scores = final_recommendation_scores.join(technology_salary_profile["salary_score"])

In [160]:
final_recommendation_scores.salary_score.fillna(0.5)

JavaScript        0.381818
Docker            0.654545
Node.js           0.536364
HTML/CSS          0.318182
AWS               0.709091
                    ...   
Play Framework    0.500000
Fortran           0.500000
SAS               0.500000
OCaml             0.500000
APL               0.500000
Name: salary_score, Length: 112, dtype: float64

In [161]:
final_recommendation_scores.head(15)

,similarity_score,salary_score
JavaScript,0.527917,0.381818
Docker,0.516275,0.654545
Node.js,0.479101,0.536364
HTML/CSS,0.474151,0.318182
AWS,0.463673,0.709091
SQL,0.455249,0.309091
TypeScript,0.452760,0.600000
MySQL,0.414426,0.263636
Bash/Shell,0.412580,0.572727
MongoDB,0.401918,0.554545


In [163]:
final_recommendation_scores["final_score"] = final_recommendation_scores.similarity_score * 0.70 + final_recommendation_scores.salary_score * 0.30

In [164]:
final_recommendation_scores.head()

,similarity_score,salary_score,final_score
JavaScript,0.527917,0.381818,0.484087
Docker,0.516275,0.654545,0.557756
Node.js,0.479101,0.536364,0.496280
HTML/CSS,0.474151,0.318182,0.427360
AWS,0.463673,0.709091,0.537299


In [165]:
final_recommendation_scores = (
    recommendation_candidates
    .rename("similarity_score")
    .to_frame()
    .join(
        technology_salary_profile[["salary_score"]],
        how="left"
    )
)

In [166]:
final_recommendation_scores["salary_score"] = (
    final_recommendation_scores["salary_score"]
    .fillna(0.5)
)

In [167]:
final_recommendation_scores.head(10)

,similarity_score,salary_score
JavaScript,0.527917,0.381818
Docker,0.516275,0.654545
Node.js,0.479101,0.536364
HTML/CSS,0.474151,0.318182
AWS,0.463673,0.709091
SQL,0.455249,0.309091
TypeScript,0.452760,0.600000
MySQL,0.414426,0.263636
Bash/Shell,0.412580,0.572727
MongoDB,0.401918,0.554545


In [168]:
similarity_weight = 0.70
salary_weight = 0.30

final_recommendation_scores["final_score"] = (
    final_recommendation_scores["similarity_score"] * similarity_weight
    + final_recommendation_scores["salary_score"] * salary_weight
)

In [169]:
final_recommendation_scores = (
    final_recommendation_scores
    .sort_values("final_score", ascending=False)
)

final_recommendation_scores.head(10)

,similarity_score,salary_score,final_score
Docker,0.516275,0.654545,0.557756
AWS,0.463673,0.709091,0.537299
Kubernetes,0.348408,0.890909,0.511158
Redis,0.381324,0.772727,0.498745
TypeScript,0.452760,0.600000,0.496932
Node.js,0.479101,0.536364,0.496280
JavaScript,0.527917,0.381818,0.484087
Terraform,0.264681,0.981818,0.479822
Homebrew,0.265790,0.972727,0.477871
Yarn,0.378220,0.690909,0.472027


In [170]:
def recommend_technologies(user_technologies, top_n=5):
    cleaned_technologies = [
        technology.strip()
        for technology in user_technologies
        if isinstance(technology, str) and technology.strip()
    ]

    valid_technologies = [
        technology
        for technology in cleaned_technologies
        if technology in technology_similarity_matrix.columns
    ]

    if not valid_technologies:
        raise ValueError(
            "Girilen teknolojilerin hiçbiri öneri motorunda bulunamadı."
        )

    similarity_scores = (
        technology_similarity_matrix[valid_technologies]
        .mean(axis=1)
        .drop(index=valid_technologies, errors="ignore")
    )

    recommendation_scores = (
        similarity_scores
        .rename("similarity_score")
        .to_frame()
        .join(
            technology_salary_profile[["salary_score"]],
            how="left"
        )
    )

    recommendation_scores["salary_score"] = (
        recommendation_scores["salary_score"]
        .fillna(0.5)
    )

    recommendation_scores["final_score"] = (
        recommendation_scores["similarity_score"] * 0.70
        + recommendation_scores["salary_score"] * 0.30
    )

    recommendations = (
        recommendation_scores
        .sort_values("final_score", ascending=False)
        .head(top_n)
        .round(4)
    )

    recommendations.index.name = "technology"

    return recommendations.reset_index()

In [171]:
recommend_technologies(
    ["Python", "FastAPI", "React.js", "PostgreSQL"],
    top_n=4
)

,technology,similarity_score,salary_score,final_score
0,Docker,0.5163,0.6545,0.5578
1,AWS,0.4637,0.7091,0.5373
2,Kubernetes,0.3484,0.8909,0.5112
3,Redis,0.3813,0.7727,0.4987


In [172]:
recommender_artifact_directory = (
    project_root / "artifacts" / "recommender"
)

recommender_artifact_directory.mkdir(
    parents=True,
    exist_ok=True
)

recommender_bundle = {
    "technology_similarity_matrix": technology_similarity_matrix,
    "technology_salary_scores": technology_salary_profile["salary_score"],
    "technology_counts": technology_counts,
    "supported_technologies": technology_similarity_matrix.columns.tolist(),
    "similarity_weight": 0.70,
    "salary_weight": 0.30,
    "minimum_salary_sample": 200,
    "model_version": "1.0.0",
}

recommender_artifact_path = (
    recommender_artifact_directory
    / "technology_recommender_bundle_v1.joblib"
)

joblib.dump(
    recommender_bundle,
    recommender_artifact_path
)

recommender_artifact_path

PosixPath('/Users/ccakir/Desktop/codepath_ai/artifacts/recommender/technology_recommender_bundle_v1.joblib')

In [173]:
loaded_recommender_bundle = joblib.load(
    recommender_artifact_path
)

loaded_recommender_bundle.keys()

dict_keys(['technology_similarity_matrix', 'technology_salary_scores', 'technology_counts', 'supported_technologies', 'similarity_weight', 'salary_weight', 'minimum_salary_sample', 'model_version'])